# Build and Test Multi-User Conversational AI Agent

Agents are systems that use an LLM as a reasoning engine to determine which actions to take and what the inputs to those actions should be. The results of those actions can then be fed back into the agent and it determines whether more actions are needed, or whether it is okay to stop.

![](https://i.imgur.com/1uVnBAm.png)

## Create Tools

Here we create two custom tools which are wrappers on top of the [Tavily API](https://tavily.com/#api) and [WeatherAPI](https://www.weatherapi.com/)

- Web Search tool with information extraction
- Weather tool

![](https://i.imgur.com/TyPAYXE.png)

In [3]:
import os
from tqdm import tqdm
from langchain.tools import tool
from markitdown import MarkItDown
from langchain_tavily import TavilySearch

import requests
from concurrent.futures import ThreadPoolExecutor, TimeoutError

tavily_tool = TavilySearch(max_results=5,
                            search_depth='advanced',
                            include_answers=False,
                            include_raw_content=True)

# certain websites won't let you crawl them unless you specify a user-agent
# pretending to be a real browser
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/112.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br"
})

md = MarkItDown(requests_session=session)

@tool
def search_web_extract_info(query:str) -> list:
    """Search the web for a query and extracts useful information from the search links"""
    print('Calling web search tool')
    results = tavily_tool.invoke(query)['results']
    docs = []
    
    def extract_content(url):
        """Helper function to extract content from a URL."""
        extracted_info = md.convert(url)
        text_title = extracted_info.title.strip()
        text_content = extracted_info.text_content.strip()
        return text_title + '\n' + text_content
    
    def fallback_doc(result):
        text_title = result['title'].strip()
        text_content = result['content'].strip()
        return f"{text_title}\n{text_content}"

    # parallelize execution of different urls
    with ThreadPoolExecutor() as executor:
        for result in tqdm(results):
            try:
                future = executor.submit(extract_content, result['url'])
                # Wait for up to 15 seconds for the task to complete
                content = future.result(timeout=15)
                docs.append(content)
            except TimeoutError:
                print(f"Extraction timed out for url: {result['url']}")
                docs.append(fallback_doc(result))
            except Exception as e:
                print(f"Error extracting from url: {result['url']} - {e}")
                docs.append(fallback_doc(result))

    return docs

@tool
def get_weather(query: str) -> list:
    """Search weatherapi to get the current weather."""
    print('Calling weather tool')
    WEATHER_API_KEY=os.getenv("WEATHER_API_KEY")
    base_url = "http://api.weatherapi.com/v1/current.json"
    complete_url = f"{base_url}?key={WEATHER_API_KEY}&q={query}"

    response = requests.get(complete_url)
    data = response.json()
    if data.get("location"):
        return data
    else:
        return "Weather Data Not Found"

/media/ultron.legacy/HDD/GitHub/ai-agent-lab/.venv/lib/python3.10/site-packages/langchain_tavily/tavily_research.py:97: UserWarning: Field name "output_schema" in "TavilyResearch" shadows an attribute in parent "BaseTool"
  class TavilyResearch(BaseTool):  # type: ignore[override, override]
/media/ultron.legacy/HDD/GitHub/ai-agent-lab/.venv/lib/python3.10/site-packages/langchain_tavily/tavily_research.py:97: UserWarning: Field name "stream" in "TavilyResearch" shadows an attribute in parent "BaseTool"
  class TavilyResearch(BaseTool):  # type: ignore[override, override]


## Test Tool Calling with LLM

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
tools = [search_web_extract_info, get_weather]

llm_with_tools = llm.bind_tools(tools)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [3]:
prompt = "Get details of Microsoft's earnings call Q4 2024"
response = llm_with_tools.invoke(prompt)
response.tool_calls

[{'name': 'search_web_extract_info',
  'args': {'query': 'Microsoft earnings call Q4 2024'},
  'id': '00c073ed-75e8-49cc-a43d-1e57932dd050',
  'type': 'tool_call'}]

In [4]:
prompt = "how is the weather in Bangalore today"
response = llm_with_tools.invoke(prompt)
response.tool_calls

[{'name': 'get_weather',
  'args': {'query': 'Bangalore'},
  'id': '30012614-7b17-4221-8ae4-c76d4a5e5e7c',
  'type': 'tool_call'}]

## Add User-Based Memory

We will now use `SQLChatMessageHistory` to store separate conversation histories per user or session.

This will be accessed by many users at the same time.

In [1]:
# removes the memory database file - usually not needed
# you can run this only when you want to remove ALL conversation histories
# ok if you get rm: cannot remove 'memory.db': No such file or directory  because initially no memory exists
!rm memory.db

rm: cannot remove 'memory.db': No such file or directory


In [2]:
from langgraph.checkpoint.sqlite import SqliteSaver

# ✅ This creates memory.db automatically (if missing)
checkpointer = SqliteSaver.from_conn_string("memory.db")

In [10]:
from langgraph.checkpoint.memory import MemorySaver  # ✅ Sync, no DB setup

checkpointer = MemorySaver()  # In-memory history (persists during session)

## Build and Test AI Agent

Now that we have defined the tools and the LLM, we can create the agent. We will be using a tool calling agent to bind the tools to the agent with a prompt. We will also add in the capability to store historical conversations as memory

In [5]:
SYS_PROMPT = """Act as a helpful assistant.
                You run in a loop of Thought, Action, PAUSE, Observation.
                At the end of the loop, you output an Answer.
                Use Thought to describe your thoughts about the question you have been asked.
                Use Action to run one of the actions available to you - then return PAUSE.
                Observation will be the result of running those actions.
                Repeat till you get to the answer for the given user query.

                Use the following workflow format:
                  Question: the input task you must solve
                  Thought: you should always think about what to do
                  Action: the action to take which can be any of the following:
                            - break it into smaller steps if needed
                            - see if you can answer the given task with your trained knowledge
                            - call the most relevant tools at your disposal mentioned below in case you need more information
                  Action Input: the input to the action
                  Observation: the result of the action
                  ... (this Thought/Action/Action Input/Observation can repeat N times)
                  Thought: I now know the final answer
                  Final Answer: the final answer to the original input question

                Tools at your disposal to perform tasks as needed:
                  - get_weather: whenever user asks get the weather of a place.
                  - search_web_extract_info: whenever user asks for specific information or if you don't know the answer.
             """

Now, we can initalize the agent with the LLM, the prompt, and the tools.

The agent is responsible for taking in input and deciding what actions to take.
Note that we are passing in the model `llm`, not `llm_with_tools`.

That is because `create_agent` will call `.bind_tools` for us under the hood.
And also execute the `tool` suggested by `llm`.

This should ideally be used with an LLM which supports tool \ function calling

In [11]:
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
tools = [search_web_extract_info, get_weather]

agent = create_agent(
            model=llm, 
            tools =tools,
            system_prompt=SYS_PROMPT,
            checkpointer=checkpointer
        )

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [7]:
def get_final_answer(result):
    """Extract final answer from any agent response."""
    try:
        final_msg = result["messages"][-1] if "messages" in result else result
        
        content = final_msg.content if hasattr(final_msg, 'content') else final_msg
        
        if isinstance(content, list) and content:
            # Gemini / structured content
            text_items = [item.get("text", "") for item in content if isinstance(item, dict) and item.get("type") == "text"]
            return text_items[0] if text_items else "No answer"
        elif isinstance(content, str):
            return content
        else:
            return str(content)[:500]  # Truncate long content
        
    except (KeyError, IndexError, AttributeError):
        return "No final answer generated"

In [12]:
# 2. USER-BASED HISTORY (3 lines total!)
def chat(user_id: str, query: str):
    config = {"configurable": {"thread_id": f"user_{user_id}"}}
    result = agent.invoke(
        {"messages": [{"role": "user", "content": query}]},
        config
    )
    return get_final_answer(result)

Querying with Agent, to call tool inorder to get info about Nvidia's Q1 2025 earnings.

In [13]:
from IPython.display import display, Markdown

query = """Summarize the key points discussed in Nvidia's Q1 2025 earnings call"""
answer = chat("user_123", query)
display(Markdown(answer))

Calling web search tool


 20%|██        | 1/5 [00:00<00:01,  2.96it/s]

Error extracting from url: https://www.investing.com/news/transcripts/earnings-call-transcript-nvidia-beats-q1-2025-expectations-stock-up-43-93CH-4069071 - 403 Client Error: Forbidden for url: https://www.investing.com/news/transcripts/earnings-call-transcript-nvidia-beats-q1-2025-expectations-stock-up-43-93CH-4069071


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
 40%|████      | 2/5 [00:02<00:03,  1.13s/it]

Error extracting from url: https://www.youtube.com/watch?v=CTx5aH3r67o - 'NoneType' object has no attribute 'strip'


 60%|██████    | 3/5 [00:03<00:02,  1.23s/it]

Error extracting from url: https://www.gurufocus.com/stock/NVDA/transcripts/2444301 - 403 Client Error: Forbidden for url: https://www.gurufocus.com/stock/NVDA/transcripts/2444301


100%|██████████| 5/5 [00:06<00:00,  1.38s/it]


Error extracting from url: https://www.businessinsider.com/nvidia-q1-earnings-call-fy26-takeaways-summary-jensen-huang-china-2025-5 - 'NoneType' object has no attribute 'strip'


Nvidia's Q1 2025 earnings call highlighted several key points:

*   **Strong Financial Performance:** Nvidia reported strong Q1 2025 results with a total revenue of $44 billion, a 69% year-over-year increase, surpassing their outlook. Data center revenue was particularly strong, growing 73% year-on-year to $39 billion.
*   **Shareholder Returns:** The company returned a record $14.3 billion to shareholders through share repurchases and cash dividends.
*   **Q2 2025 Outlook:** For Q2 2025, Nvidia projects total revenue of $45 billion, plus or minus 2%, with anticipated modest sequential growth across all platforms.
*   **AI Infrastructure Dominance:** Nvidia is solidifying its position as the core infrastructure provider for global AI, likening AI to a utility like electricity. The build-out of "AI factories" is a significant revenue driver.
*   **Surge in AI Inference:** There's an "explosion" in demand for AI inference, with customers rapidly scaling their inference output using Nvidia's technology. New products like the B300 GPU and Llama-based AI models demonstrate substantial performance improvements.
*   **Networking as a "Silent Giant":** The networking segment, particularly NVLink and Spectrum X, is crucial for powering AI factories in major cloud environments like Azure and Google Cloud. Networking revenue increased 64% quarter-over-quarter, with Spectrum X alone annualizing over $8 billion.
*   **Growth in Gaming and AI PCs:** Gaming revenue reached $3.8 billion, a 48% sequential increase, driven by the rising popularity of AI-capable laptops.
*   **Sovereign AI:** AI is being viewed as a foundational national infrastructure, with countries globally investing in building their own AI capabilities at scale.
*   **Challenges in China:** Export restrictions continue to impact Nvidia's business in China, leading to a $2.5 billion disruption in Q1 product shipments and an anticipated $8 billion revenue loss for the next quarter.
*   **Expansion into Industrial AI and Robotics:** Nvidia is seeing strong adoption of industrial AI and robotics, forming partnerships and developing open-source foundation models for humanoid robots.
*   **Investor Outlook:** The call reinforced that AI is the essential infrastructure layer across all major industries, presenting investment opportunities in areas like cloud and data infrastructure, semiconductors, enterprise AI, industrial automation, next-gen mobility, and clean energy.

Querying with Agent, to call tool inorder to get info about Intel's Q1 2025 earnings.

In [14]:
from IPython.display import display, Markdown

query = """Summarize the key points discussed in Intel's Q1 2025 earnings call"""
answer = chat("user_123", query)
display(Markdown(answer))

Calling web search tool


  0%|          | 0/5 [00:00<?, ?it/s]Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
 20%|██        | 1/5 [00:01<00:05,  1.49s/it]

Error extracting from url: https://www.youtube.com/watch?v=Gm28clo6y0Q - 'NoneType' object has no attribute 'strip'


 80%|████████  | 4/5 [00:09<00:02,  2.23s/it]

Error extracting from url: https://d1io3yog0oux5.cloudfront.net/_0e36ec7e022fbe9865750d7112d95e8d/intel/db/887/9128/prepared_remarks/Q1-2025-Earnings-Call-Script-FINAL-FINAL.pdf - File conversion failed after 1 attempts:
 - PdfConverter threw MissingDependencyException with message: PdfConverter recognized the input as a potential .pdf file, but the dependencies needed to read .pdf files have not been installed. To resolve this error, include the optional dependency [pdf] or [all] when installing MarkItDown. For example:

* pip install markitdown[pdf]
* pip install markitdown[all]
* pip install markitdown[pdf, ...]
* etc.



100%|██████████| 5/5 [00:10<00:00,  2.17s/it]


Intel's Q1 2025 earnings call revealed a mixed financial picture, with the company surpassing some expectations but providing cautious forward-looking guidance. Here are the key points:

**Financial Performance (Q1 2025):**

*   **Revenue:** Intel reported $12.7 billion in revenue, which was at the high end of their guidance and slightly above analyst expectations.
*   **Non-GAAP Gross Margin:** Achieved 39.2%, approximately 3 percentage points above guidance.
*   **Earnings Per Share (EPS):** Reported $0.13, significantly exceeding the breakeven EPS guidance.
*   **Intel Products Revenue:** $11.8 billion, a 10% sequential decrease.
*   **Intel Foundry Revenue:** $4.7 billion, an 8% sequential increase.
*   **Intel Foundry Operating Loss:** Remained relatively flat quarter-over-quarter at $2.3 billion.
*   **Operating Cash Flow:** $800 million.
*   **Cash Balance:** $21 billion.

**Forward-Looking Guidance (Q2 2025 and Beyond):**

*   **Q2 Revenue Guidance:** Projected to be between $11.2 billion and $12.4 billion, lower than analyst estimates.
*   **Q2 Gross Margin Guidance:** Approximately 36.5%.
*   **Q2 EPS Guidance:** Breakeven.
*   **Operating Expense Target (2025):** Reduced to $17 billion (from a previous target of $17.5 billion).
*   **Operating Expense Target (2026):** Targeted at $16 billion.
*   **Gross Capital Expenditure (CapEx) Target (2025):** Reduced to $18 billion (from a previous target of $20 billion).

**Key Strategic Insights and Challenges:**

*   **New Leadership and Restructuring:** This was the first earnings report under new CEO Lip-Bu Tan. The company plans to simplify operations, reduce bureaucracy, and empower smaller teams, which will involve job cuts, particularly for managers.
*   **Macroeconomic Uncertainty:** Intel cited "elevated uncertainty" due to the macro environment, including shifting trade policies and persistent inflation, increasing the probability of an economic slowdown.
*   **AI Focus:** Intel is strategically adjusting its product roadmap to optimize for emerging AI workloads and is actively partnering to optimize silicon and software for AI, including new architectures for edge and inference applications.
*   **Foundry Business:** The company is prioritizing ramping internal customers for its foundry business (e.g., Panther Lake) and building trust with external foundry clients, focusing on process technology improvements (like the 18A process).
*   **Product Mix and Demand:** There is a demand for older generation products like Raptor Lake due to macroeconomic factors and consumer price sensitivity, while newer products like Lunar Lake are expected to have margin pressures initially. Panther Lake, expected in 2026, is anticipated to improve margins.
*   **Capacity Constraints:** Intel is facing ongoing capacity constraints in Intel 7, which are expected to persist.
*   **Competitive Pressure:** The company acknowledges facing competitive pressure in its core markets, impacting product mix and margins.

Get the comparision between Nvidia VS Intel Q1 2025 earning.
And now we have history parameter, so our agent could recall the past conversations.

In [15]:
from IPython.display import display, Markdown

query = """which company's future outlook looks to be better?"""
answer = chat("user_123", query)
display(Markdown(answer))

Based on the key points from both companies' Q1 2025 earnings calls, **Nvidia's future outlook appears to be significantly stronger than Intel's.**

Here's a comparison of the key factors:

**Nvidia:**
*   **Strong Growth:** Reported robust Q1 2025 revenue of $44 billion (up 69% year-over-year) and specifically high growth in its Data Center segment (up 73% year-over-year to $39 billion).
*   **Positive Outlook:** Projected continued modest sequential growth for Q2 2025, with an expected revenue of $45 billion.
*   **Market Leadership:** Dominates the booming AI market, positioning itself as the foundational infrastructure provider for AI.
*   **Diversified AI Applications:** Seeing massive demand for AI inference, significant growth in networking solutions (NVLink & Spectrum X), and strong performance in gaming and AI PCs.
*   **Innovation:** Continues to drive advancements with new GPUs and AI models, yielding substantial performance improvements.
*   **Main Challenge:** Geopolitical factors, particularly US export restrictions to China, which significantly impacted Q1 shipments ($2.5 billion disruption) and are expected to cause an $8 billion revenue loss in the next quarter.

**Intel:**
*   **Mixed Performance:** While Q1 2025 revenue ($12.7 billion) and EPS ($0.13) beat guidance, Intel Products revenue was down 10% sequentially.
*   **Cautious Guidance:** Issued a weaker Q2 revenue guidance ($11.2 billion to $12.4 billion, below analyst estimates) and predicted breakeven EPS, citing elevated macroeconomic uncertainty and fluid trade policies.
*   **Foundry Business Challenges:** The Intel Foundry segment reported a significant operating loss of $2.3 billion, which remained flat quarter-over-quarter, indicating it's still in a heavy investment and development phase.
*   **Restructuring and Cost Cutting:** Under new CEO Lip-Bu Tan, the company is implementing aggressive operational and capital expense reductions, including management layer removals and job cuts, to improve efficiency.
*   **Competitive Pressure:** Acknowledges facing ongoing competitive pressure in its core markets and a need to regain market share, especially in AI.
*   **Capacity Constraints:** Facing capacity constraints in Intel 7 manufacturing.

**Conclusion:**

Nvidia is currently in a strong market leadership position, riding the wave of explosive demand in AI, which is driving substantial revenue growth and positive future projections despite export restrictions. Their products and ecosystem are at the forefront of a transformative technological shift.

Intel, conversely, is undergoing a significant turnaround. While showing signs of beating some internal targets, it faces a more challenging macroeconomic environment, intense competition, and the difficult task of revitalizing its core businesses while simultaneously building out a profitable foundry and competitive AI offerings. The weak Q2 guidance and persistent foundry losses highlight the arduous path ahead.

Therefore, **Nvidia's future outlook appears significantly brighter** due to its dominant position in a high-growth market, strong financial performance, and clear strategic direction, compared to Intel's ongoing restructuring and more uncertain path to growth and profitability.

Querying with Agent, to call tool inorder to get info about Bangalore weather.

In [16]:
from IPython.display import display, Markdown

query = """how is the weather in Bangalore today?
           show detailed statistics
        """
answer = chat("user_456", query)
display(Markdown(answer))

Calling weather tool


The current weather in Bangalore, India is:
*   **Condition:** Mist
*   **Temperature:** 22.4°C (72.3°F), feels like 24.5°C (76.2°F)
*   **Humidity:** 69%
*   **Wind:** 11.9 kph (7.4 mph) from the East (86 degrees)
*   **Pressure:** 1017 mb (30.03 in)
*   **Precipitation:** 0 mm (0 in)
*   **Visibility:** 5 km (3 miles)
*   **Cloudiness:** 50%
*   **UV Index:** 0
*   **Dewpoint:** 9.6°C (49.3°F)
*   **Gust:** 21.6 kph (13.4 mph)
*   **Heat Index:** 20.2°C (68.4°F)
*   **Windchill:** 20.2°C (68.4°F)
*   **Last Updated:** 2025-12-30 22:00 (local time)

Querying with Agent, to call tool inorder to get info about Dubai weather.

In [17]:
from IPython.display import display, Markdown

query = """how is the weather in Dubai today?
           show detailed statistics
        """
answer = chat("user_456", query)
display(Markdown(answer))

Calling weather tool


The current weather in Dubai, United Arab Emirates is:
*   **Condition:** Patchy rain nearby
*   **Temperature:** 23.1°C (73.6°F), feels like 24.9°C (76.9°F)
*   **Humidity:** 47%
*   **Wind:** 40 kph (24.8 mph) from the West-Northwest (294 degrees)
*   **Pressure:** 1016 mb (30.00 in)
*   **Precipitation:** 0.01 mm (0 in)
*   **Visibility:** 10 km (6 miles)
*   **Cloudiness:** 0%
*   **UV Index:** 0
*   **Dewpoint:** 13.4°C (56.2°F)
*   **Gust:** 54.1 kph (33.6 mph)
*   **Heat Index:** 24.6°C (76.2°F)
*   **Windchill:** 22.1°C (71.8°F)
*   **Last Updated:** 2025-12-30 20:30 (local time)

Get the comparision between Bangalore and Dubai weather.
As we have history parameter, so our agent could recall the past conversations.

In [18]:
from IPython.display import display, Markdown

query = """which city is hotter?
        """
answer = chat("user_456", query)
display(Markdown(answer))

Dubai is hotter than Bangalore. The current temperature in Dubai is 23.1°C, while in Bangalore it is 22.4°C.

User-Based info

In [19]:
from IPython.display import display, Markdown

query = """which city is colder?
        """
answer = chat("user_123", query)
display(Markdown(answer))

Please tell me which cities you would like to compare.

In [20]:
from IPython.display import display, Markdown

query = """which city is colder?
        """
answer = chat("user_456", query)
display(Markdown(answer))

Bangalore is colder than Dubai. The current temperature in Bangalore is 22.4°C, while in Dubai it is 23.1°C.